# Setting up the libraries
I'm using chatGPT to help me translate

In [10]:
from systole.detection import ppg_peaks
import pandas as pd
import numpy as np
import matplotlib as plt
import os
import re
import pingouin as pg
pd.options.mode.copy_on_write = True # to remove the weird behaviour that python changes the original object
base_path='/Users/jacquelinescholl/Documents/Experiments/Tim_Palmer/FMRI_study'

We're going to preprocess exactly as in the original code. Then we will later compare this to what I do in R, because I don't quite understand how the grouping works. The reason we use Python is to have the heart peak detection, so I will also store the files with that.

In [11]:
folders = [f for f in os.listdir(os.path.join(base_path, 'Data', 'Cardio_rawData')) if os.path.isdir(os.path.join(base_path, 'Data', 'Cardio_rawData', f))]

signal_quality = pd.DataFrame()
behav_full = pd.DataFrame()
files_missing = []

for ifolder in folders:
    this_file = [f for f in os.listdir(os.path.join(base_path, 'Data', 'Cardio_rawData', ifolder)) if re.search('_signal.txt$', f)]
    this_file_path = os.path.join(base_path, 'Data', 'Cardio_rawData', ifolder, this_file[0]) if this_file else None
    
    this_file_behav= [f for f in os.listdir(os.path.join(base_path, 'Data', 'Cardio_rawData', ifolder)) if re.search('_final.txt$', f)]
    this_file_path_behav = os.path.join(base_path, 'Data', 'Cardio_rawData', ifolder, this_file_behav[0]) if this_file_behav else None
    
    if this_file:
        # the signal
        df = pd.read_csv(this_file_path)
        nonBRIC_ID, session = re.split('_', ifolder)
        df['nonBRIC_ID'] = nonBRIC_ID
        df['session'] = session
        #signal_df['Time'] = np.arange(0, len(signal_df))/1000 # Create time vector
        signal_quality = pd.concat([signal_quality, df], ignore_index=True)

        # the behaviour
        df = pd.read_csv(this_file_path_behav)
        df['nonBRIC_ID'] = nonBRIC_ID
        df['session'] = session
        behav_full = pd.concat([behav_full, df], ignore_index=True)


    else:
        files_missing.append(ifolder)



Now for each person, compute the outlier measures

In [14]:

for i2,iID in enumerate(signal_quality.nonBRIC_ID.unique()):
    this_signal_byID =  signal_quality[(signal_quality['nonBRIC_ID'] == iID)]
    for i3, iSess in enumerate(this_signal_byID.session.unique()):
        this_signal_byIDbySess = this_signal_byID[(this_signal_byID['session'] == iSess)]
        clean_df = behav_full[(behav_full['nonBRIC_ID']==iID) & (behav_full['session']==iSess)] #behav_full.copy()
        clean_df['HeartRateOutlier'] = np.zeros(len(clean_df), dtype='bool')
        drop, bpm_std, bpm_df = [], [], pd.DataFrame([])
        outliers= pd.DataFrame()
        for i, trial in enumerate(this_signal_byIDbySess.nTrial.unique()):
            color = '#c44e52' if (i % 2) == 0 else '#4c72b0'
    #       this_df = signal_quality[signal_quality.nTrial == trial & signal_quality.nonBRIC.ID=iID]  # Downsample to save memory
            this_signal_byIDbySessbyTrial = this_signal_byIDbySess[(this_signal_byIDbySess['nTrial'] == trial)]  #Downsample to save memory

            signal, peaks = ppg_peaks(this_signal_byIDbySessbyTrial.signal, sfreq=1000,clipping='false') # normally clipping=T, but there is an issue here in find_clipping that if it only detects one max value, it does not output a threshold, so the subsequent code does not work
            bpm = 60000/np.diff(np.where(peaks)[0])

            bpm_df = pd.concat(
                [
                    bpm_df,
                    pd.DataFrame({'bpm': bpm, 'nEpoch': i, 'nTrial': trial})
                ]
            ) # data frame with the peack and the rate at that point
        
        # Check for outliers in the absolute value of RR intervals
        for e, t in zip(bpm_df.nEpoch[pg.madmedianrule(bpm_df.bpm.to_numpy())].unique(),
                        bpm_df.nTrial[pg.madmedianrule(bpm_df.bpm.to_numpy())].unique()):
            drop.append(e)
            print(t)
            #I'm not sure this is correct in original code - in my python version, it just appends to the df:
            # clean_df.loc[t, 'HeartRateOutlier'] = True
            clean_df.loc[clean_df['nTrials'] == t, 'HeartRateOutlier'] = True

        # Check for outliers in the standard deviation values of RR intervals
        for e, t in zip(np.arange(0, bpm_df.nTrial.nunique())[pg.madmedianrule(bpm_df.copy().groupby(['nTrial', 'nEpoch']).bpm.std().to_numpy())],
                        bpm_df.nTrial.unique()[pg.madmedianrule(bpm_df.copy().groupby(['nTrial', 'nEpoch']).bpm.std().to_numpy())]):
            if e not in drop:
                drop.append(e)
                #clean_df.loc[t, 'HeartRateOutlier'] = True
                clean_df.loc[clean_df['nTrials'] == t, 'HeartRateOutlier'] = True

        # For checks in R: write files also with the indices from the detection algorithm
        rows_tagged = bpm_df.copy()
        rows_tagged['mean_madmedian']=pg.madmedianrule(bpm_df.bpm.to_numpy())
        t=rows_tagged.groupby('nTrial')['mean_madmedian'].any()
        trials_tagged= pd.DataFrame()
        trials_tagged['nTrial']=bpm_df.nTrial.unique()
        trials_tagged['sd_madmedian']=pg.madmedianrule(bpm_df.copy().groupby(['nTrial', 'nEpoch']).bpm.std().to_numpy())
        trials_tagged=pd.merge(trials_tagged,t,how='inner',on=['nTrial'])

        bpm_df['nonBRIC_ID']  = iID
        bpm_df['session'] = iSess
        # write the files: 
        
        #np.savetxt(os.path.join(base_path,'Data_preprocessed','Cardio_fromPython',
         #                       'bpm'+iID+'_'+iSess+'.txt'), (bpm_df), delimiter=',',fmt='%s')
        bpm_df.to_csv(os.path.join(base_path,'Data_preprocessed','Cardio_fromPython','bpm_'+iID+'_'+iSess+'.txt'))
        trials_tagged.to_csv(os.path.join(base_path,'Data_preprocessed','Cardio_fromPython','trialsWithOutlierTags_'+iID+'_'+iSess+'.txt'))    
        clean_df.to_csv(os.path.join(base_path,'Data_preprocessed','Cardio_fromPython','pythonTaggedAllOutliersDf_'+iID+'_'+iSess+'.txt'))    


11
20
98
1
8
10
0
1
7
44
74
86
114
3
19
20
26
33
35
41
44
61
62
67
80
82
83
87
95
102
104
109
113
118
9
56
58
59
75
116
7
29
48
66
15
41
52
93
0
69
88
112
34
41
45
46
60
63
78
101
103
108
1
2
19
59
72
85
88
113
114
0
19
21
82
83
84
85
90
0
24
27
80
81
82
98
105
115
0
28
38
61
84
95
99
102
103
16
19
41
58
62
66
83
96
99
102
119
10
32
39
43
59
100
101
111
115
6
15
16
21
42
56
57
81
85
89
101
116
0
24
27
36
45
67
82
92
96
98
107
16
21
41
58
62
63
71
77
78
116
117
0
2
4
7
18
65
6
19
21
51
103
104
18
21
22
23
36
39
89
1
6
10
55
77
80
110
117
0
7
9
17
33
55
58
62
77
98
104
95
96
97
113
41
88
89
93
109
31
39
85
114
6
46
48
54
71
73
101
110
20
63
82
83
85
96
100
101
108
116
8
61
63
73
87
88
101
105
107
116
119
5
8
14
21
28
35
39
40
43
59
64
71
101
2
3
102
31
48
49
70
71
87
92
62
68
114
117
15
16
17
21
101
112
18
26
44
45
100
21
35
53
88
91
97
101
19
78
17
29
52
59
63
77
89
102
115
0
5
24
41
53
55
79
95
105
115
26
81
90
115
0
11
21
35
37
39
44
51
52
56
65
72
106
112
